# ERα–SRC2 ligand MD: analysis, figures and end-point energies

This notebook reads the finished production runs from Google Drive: 9 systems × 3 independent replicas × 100 ns.
It computes stability and interaction measures and MM/GBSA end-point energies, and draws the publication figures.
Each stage saves its output to Drive (`eralpha-antidepressant-md/analysis/`) and skips work that is already done, so the notebook
can be rerun after a disconnect without losing anything.

| Output | What it shows |
|---|---|
| `Fig_MD_rmsd` | ligand RMSD per replica (mean ± SEM band), symmetry-corrected |
| `Fig_MD_interactions` | pose retention, ΔΔE versus estradiol at ε_in = 1, 2, 4 and PB, interaction occupancy |
| `FigS*` | receptor and helix-12 RMSD, RMSF, SRC2 coactivator, ligand distances, convergence, decomposition, sensitivity, run QC |
| `tables/main_table.csv` | the summary table: end-point energies, retention and anchor occupancy per system |

**Statistics.** The unit of replication is the replica (*n* = 3). Frames inside one simulation are correlated, so they
are averaged per replica first. Uncertainties are the SEM of the three replica means, with t-based 95 % confidence
intervals.

## 0 · Setup
Parameters: change `KEYS` to restrict the run to some systems (e.g. `['estradiol_rep1']` for a pilot).

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/eralpha-antidepressant-md"   # written by the production run
REPO_URL   = "https://github.com/masoudrezaei082-afk/eralpha-antidepressant-md.git"
REPO       = "/content/md-workflow"
ENV        = "/content/mamba/envs/tools"            # pinned GROMACS 2026.3 / AmberTools 26 / MDAnalysis env
KEYS       = None                                    # None = all 27 replicas
NPROC      = 2                                       # MPI ranks for gmx_MMPBSA (free Colab CPU: 2 vCPU)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.isdir(f"{DRIVE_ROOT}/runs"), "eralpha-antidepressant-md/runs not found on Drive"
if not os.path.isdir(REPO):
    !git clone -q --depth 1 {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q

The analysis runs in a pinned environment (the same lock file as the simulation container), created with
micromamba. It takes about 10 minutes the first time and is skipped when it already exists. The notebook kernel only
orchestrates the work and displays the results; every computation runs in this environment.

In [ ]:
import subprocess, os, json, time
if not os.path.exists(f"{ENV}/bin/gmx_MMPBSA"):
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/2.3.2 | tar -xj -C /content bin/micromamba
    !MAMBA_ROOT_PREFIX=/content/mamba /content/bin/micromamba create -q -y -p {ENV} --file {REPO}/docker/tools-env.lock.txt
    !{ENV}/bin/pip install -q "gmx_MMPBSA==1.7.0" "spyrmsd==0.9.0" pyarrow openpyxl
RUN_ENV = dict(os.environ, MPLBACKEND="Agg", AMBERHOME=ENV, PATH=f"{ENV}/bin:" + os.environ["PATH"],
               PYTHONPATH=f"{REPO}/analysis", MDANA_DRIVE=DRIVE_ROOT)

def envpy(args, background=None, timeout=None):
    """Run `python <args>` in the analysis env; background=<log path> detaches it (survives this cell)."""
    if background:
        cmd = f"nohup {ENV}/bin/python {args} > {background} 2>&1 &"
        subprocess.Popen(cmd, shell=True, env=RUN_ENV, start_new_session=True)
        return f"started; log: {background}"
    r = subprocess.run(f"{ENV}/bin/python {args}", shell=True, env=RUN_ENV, capture_output=True, text=True,
                       timeout=timeout)
    if r.returncode:
        raise RuntimeError(r.stderr[-3000:])
    return r.stdout

print(envpy("-m mdana.stats"))   # unit test of the statistics on hand-checked toy data
keys = " ".join(KEYS) if KEYS else ""
ANALYSIS = f"{DRIVE_ROOT}/analysis"

## 1–3 · Trajectories and structural measures

For each replica:

1. **Join.** The 100-ns trajectory was written in parts. A part interrupted by a crash overlaps the next one; the
   overlap is dropped. The joined trajectory must hold 10,001 frames (one every 10 ps from 0 to 100 ns).
2. **Clean up the periodic box.** The simulation box repeats in every direction, so a molecule can appear split, or
   sit next to a copy of the receptor rather than the receptor itself. Every molecule is made whole, and the SRC2
   peptide and the ligand are moved next to the receptor. Each frame is then superimposed on the binding-pocket
   Cα atoms of the starting structure.
3. **Measure.** Each measure is described in plain language for the reader:
   * **Ligand RMSD:** how far, on average, the ligand's heavy atoms have moved from the starting (docked or
     crystal) pose, after the pocket is superimposed. It is *symmetry-corrected*, so swapping two equivalent atoms
     (for example a flipped ring) does not count as movement.
   * **Pose retention:** the share of the 20–100 ns frames in which that RMSD stays below 2 Å, i.e. the ligand is
     still in essentially its starting pose. Higher means more stable.
   * **Out of contact:** frames where no ligand atom is within 4 Å of the protein, meaning the ligand has left the
     pocket.
   * **Receptor backbone RMSD, Cα RMSF, helix-12 RMSD:** overall structural drift, residue flexibility, and the
     position of helix 12, which controls coactivator binding.
   * **SRC2 RMSD and charge clamp:** whether the coactivator peptide stays bound in its groove. The clamp is
     Lys362 and Glu542 holding the peptide's backbone ends.
   * **Contacts, H-bonds, salt bridges:** the share of frames in which each residue touches, H-bonds or forms an
     ion pair with the ligand. The key anchors are Glu353, Arg394 and His524.

The first 20 ns of production are excluded from every average (equilibration), and are shaded grey in the figures.

**What these measures do *not* show.** They describe whether a pose is stable at equilibrium on a 100-ns scale. They
do not give binding kinetics, residence times or off-rates.

This stage runs in the background (about 10 min per replica). Re-run the status cell until all replicas are done.

In [ ]:
print(envpy(f"-m mdana.pipeline structure {'--keys ' + keys if keys else ''}", background="/content/structure.log"))

In [ ]:
p = f"{ANALYSIS}/structure_progress.json"
if os.path.exists(p):
    s = json.load(open(p))
    print(f"done {len(s['done'])}/{s['total']}, current: {s.get('current')}, failed: {[f['id'] for f in s['failed']]}")
!tail -n 3 /content/structure.log

## 4 · MM/GBSA and MM/PBSA end-point energies

MM/GBSA estimates how much more favourable the energy of the complex is than that of the separated receptor and
ligand, using frames from the simulation and an implicit-solvent model. It is an **end-point energy without entropy**,
not a binding free energy or an affinity. It is used here only to *rank* ligands against estradiol (the native
agonist) and a decoy (the negative control).

Settings, each justified in the committed input files `analysis/mmpbsa_gb.in` and `mmpbsa_pb.in`:

* **Frames:** 20–100 ns, one every 0.5 ns, giving 160 frames per replica and 480 per system.
* **GB model:** GB-neck2 (`igb = 8`) with its matching mbondi3 radii.
* **Salt:** 0.15 M, the same as in the explicit-solvent simulations.
* **Interior dielectric ε_in = 1, 2 and 4.** The ranking is reported only where it holds at all three values.
* **PB cross-check:** a Poisson–Boltzmann calculation at ε_in = 2 on every third frame.
* **Per-residue decomposition:** at ε_in = 2 only.

The 96 jobs run one after another in a background process, with ε_in = 2 first. A disconnected runtime resumes at
the next unfinished job.

In [ ]:
print(envpy(f"-m mdana.mmgbsa run --np {NPROC} {'--keys ' + keys if keys else ''}", background="/content/mmgbsa.log"))

In [ ]:
print(envpy("-m mdana.mmgbsa status"))
!grep -E "ΔTOTAL|FAILED" /content/mmgbsa.log | tail -n 5

## 5 · Statistics and tables

* **System values:** per-replica means are combined into a system mean, its SEM and a t-based 95 % CI (n = 3).
* **ΔΔE versus estradiol:** the difference of means with Welch's 95 % CI.
* **Stating an ordering:** a ligand is called more or less favourable than estradiol only if the CI excludes zero
  at *every* ε_in and in PB. Otherwise the result is reported as "not resolved".
* **Convergence:**
  * Block averaging (Flyvbjerg–Petersen) within each replica.
  * Running means of ΔE.
  * A comparison of the first and second halves of the run.
* **Ranking agreement:** Spearman's ρ between the rankings at different ε_in, and between GB and PB.
* **Old versus new:** a comparison with the original single-run results.
* **Clustering:** ligand poses are clustered (GROMOS, 2 Å). The centroid complexes are saved as PDB files for the
  structure figures.

In [ ]:
print(envpy("-c 'from mdana import aggregate as A; t, s = A.run(); import json; print(json.dumps(s, indent=1, default=float))'"))
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 30)
pd.read_csv(f"{ANALYSIS}/tables/main_table.csv")

In [ ]:
for name in ("ddE_vs_estradiol", "structure_per_system", "old_vs_new", "convergence"):
    f = f"{ANALYSIS}/tables/{name}.csv"
    if os.path.exists(f):
        print(name); display(pd.read_csv(f).round(3))

## 6 · Figures

All figures share one style:
* Liberation Sans, metrically identical to Arial, at 7 pt.
* Springer Nature column widths.
* One colour-blind-safe colour per system, the same in every figure.
* Each figure is saved as a PDF with embedded fonts, an SVG and a 600-dpi PNG, to Drive `analysis/figures/`.

**Main figures** (replace Fig. 6 and Tables 1–2):
* `Fig_MD_rmsd`: ligand RMSD for each replica, with the mean ± SEM;
* `Fig_MD_interactions`: a, pose retention; b, ΔΔE versus estradiol at each ε_in and in PB; c, residue contact and H-bond occupancy.

**SI figures:**
* S1, receptor and helix-12 RMSD;
* S2, RMSF;
* S3, SRC2 and the charge clamp;
* S4, ligand position;
* S5, convergence;
* S6, per-residue decomposition;
* S7, ε_in and GB/PB sensitivity;
* S8, thermodynamic quality control.

In [ ]:
print(envpy("-c 'from mdana import figures as F; F.all_figures()'"))
from IPython.display import Image, display
import glob
for f in ["Fig_MD_rmsd", "Fig_MD_interactions"] + sorted(os.path.basename(p)[:-4] for p in glob.glob(f"{ANALYSIS}/figures/FigS*.png")):
    print(f); display(Image(f"{ANALYSIS}/figures/{f}.png", width=900))

In [ ]:
!for f in {ANALYSIS}/figures/Fig_MD_*.pdf; do pdffonts $f; done

## 7 · Interpretation guide

The results are interpreted with these rules:

* A difference is described as a difference only when its 95 % CI excludes zero at every ε_in. Otherwise the text
  says "comparable within replica spread".
* MM/GBSA values are called *end-point energies* and are used only for ranking, never as affinities.
* Stability statements are limited to "the docked pose was retained (or not) over 100 ns in *k* of 3 replicas". They
  never claim kinetics or residence time.